# Soup GRPO task: step 5, DFlash follow-up with the trained adapters (Kaggle T4)

The first DFlash run (soup_dflash) had no adapters attached, so it ran on the base model only.
This run does the parts that need them, plus the batch-48 sweep that crashed (bug E19, fixed).

**Before running:**
1. Settings: **Accelerator: GPU T4 x2** and **Internet: On**.
2. **Add Input -> Your Work -> Notebooks:** add **soup_train_main** AND **soup_train_control**.
   Required: cell 8 stops the run at once if either adapter is missing (so no GPU time is wasted).
3. **Save Version -> Save & Run All (Commit)**, close the tab. Takes about **1.5 hours**. When it shows Complete,
   open its **Output** tab and download **`soup_results.zip`**.

Pre-registered in `predictions/PREDICTIONS.md`, "Speed A/B and DFlash" (unchanged).
7. Download the DFlash drafter and find the adapters. 8. Stop if an adapter is missing.
9. **H21 + H7:** greedy, 96 held-out puzzles: main adapter (with the plain run), control adapter.
10. **H23, batch 48 only** (8 / 16 / 32 were measured in soup_dflash).
11. **H22 with the main adapter** (soup_dflash ran it on the base model).

In [ ]:
# 1. Unpack code + data (git 8969abd, bundle sha256 53a89ef108627384)
import base64, hashlib, io, json, os, subprocess, sys, tarfile, time
from pathlib import Path
LOCAL_TEST = os.environ.get("SOUP_LOCAL_TEST") == "1"   # only for testing this notebook on a laptop
WORK = Path(os.environ.get("SOUP_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == "53a89ef10862738437c13dc82e39a7949f73bd7d9c98d6fa2fc6092badaa9bd8", "bundle corrupted"
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(WORK, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
REPO = WORK / "soup-grpo"
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
man = json.loads((REPO / "data/MANIFEST.json").read_text())
for f, info in man["files"].items():
    assert hashlib.sha256((REPO / "data" / f).read_bytes()).hexdigest() == info["sha256"], f
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
(REPO / "results/pilot").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| data files verified:", len(man["files"]), "| bundle git rev 8969abd")

In [ ]:
# helper: run a command, stream its output live, keep a copy in results/logs
def run(cmd, log=None, check=True, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    t0 = time.time()
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    lines = []
    for line in p.stdout:
        if "Warning" in line or "warn(" in line:
            continue
        print(line, end="", flush=True)
        lines.append(line)
    p.wait()
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[exit {p.returncode}, {time.time() - t0:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p.returncode

PY = sys.executable
STOP = None   # set by the first failing step; later steps are skipped, results are still packed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU, as in the task
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less cache fragmentation (pilot 1: 2 GiB)
MODEL = os.environ.get("SOUP_MODEL") or ("Qwen/Qwen3-0.6B" if LOCAL_TEST else "Qwen/Qwen3-4B")   # SOUP_MODEL: local tests only
DEVICE = "cpu" if LOCAL_TEST else "cuda"

In [ ]:
# 2. Environment + dependencies
if not LOCAL_TEST:
    run([PY, "-m", "pip", "install", "-q", "transformers>=4.51", "safetensors>=0.4", "huggingface_hub"], check=False)
    TS = time.strftime("%Y%m%d-%H%M%S")
    env_txt = subprocess.run("nvidia-smi; nvidia-smi --query-gpu=name,memory.total,driver_version,"
                             "pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max "
                             "--format=csv; free -g; nproc; df -h /kaggle/working",
                             shell=True, capture_output=True, text=True).stdout
    import torch, transformers
    env_txt += f"\ntorch {torch.__version__} cuda {torch.version.cuda} transformers {transformers.__version__}"
    env_txt += f"\ndevice {torch.cuda.get_device_name(0)} capability {torch.cuda.get_device_capability(0)}"
    env_txt += f" bf16_supported {torch.cuda.is_bf16_supported()}"
    (REPO / f"results/logs/env_{TS}.txt").write_text(env_txt)
    print(env_txt)

In [ ]:
# 3. Session monitor (nvidia-smi + host RAM every second) and the 8 GB ballast
if STOP is None:
    try:
        if not LOCAL_TEST:
            TS = time.strftime("%Y%m%d-%H%M%S")
            smi_log = open(REPO / f"results/logs/nvsmi_{TS}.csv", "w")
            monitor = subprocess.Popen(["nvidia-smi", "--query-gpu=timestamp,memory.used,memory.total,utilization.gpu,"
                                        "pcie.link.gen.current,pcie.link.width.current,temperature.gpu,power.draw,clocks.sm",
                                        "--format=csv", "-l", "1"], stdout=smi_log, stderr=subprocess.STDOUT)
            host_mon = subprocess.Popen(["bash", "-c", f"while true; do echo \"$(date -Is),$(free -m | awk '/Mem:/ {{print $3\",\"$7}}')\""
                                         f" >> {REPO}/results/logs/hostmem_{TS}.csv; sleep 1; done"])
            ballast = subprocess.Popen([PY, "scripts/gpu/ballast.py", "--cap-mib", "8192"],
                                       stdout=open(REPO / "results/logs/ballast.out", "w"), stderr=subprocess.STDOUT)
            for _ in range(120):
                if (REPO / "results/logs/ballast.json").exists():
                    break
                time.sleep(1)
            BALLAST = json.loads((REPO / "results/logs/ballast.json").read_text())
            print("ballast:", BALLAST)
            assert BALLAST["free_left_mib"] <= 8192, "cap not in place"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 4. Model download (~8 GB) and correctness of the engine on this GPU
if STOP is None:
    try:
        t0 = time.time()
        from soupgrpo.runtime import resolve_model_dir
        print("downloading", MODEL, "(~8 GB, a few minutes; no output until done) ...", flush=True)
        print("model at", resolve_model_dir(MODEL), f"({time.time() - t0:.0f}s)")
        run([PY, "-m", "pytest", "-q", "-x"], log="pytest.txt", check=False)
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/cuda_equivalence.py"], log="cuda_equivalence.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 5. PCIe bandwidth (pinned vs pageable, one 4B layer = 202 MB) + CUDA context cost
if STOP is None:
    try:
        if not LOCAL_TEST:
            run([PY, "scripts/gpu/pcie_bench.py"], log="pcie.txt")
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 7. DFlash drafter (weights + config only; none of its code is run) and the trained adapters
if STOP is None:
    try:
        import shutil, zipfile
        if LOCAL_TEST:
            from soupgrpo.dflash import write_random_drafter
            from soupgrpo.runtime import resolve_model_dir
            DRAFTER = write_random_drafter(WORK / "drafter", resolve_model_dir(MODEL), 2, (1, 2), 16, 7)
        else:
            from huggingface_hub import hf_hub_download
            for f in ("config.json", "model.safetensors"):
                DRAFTER = Path(hf_hub_download("z-lab/Qwen3-4B-DFlash-b16", f)).parent
        print("drafter at", DRAFTER, [p.name for p in DRAFTER.iterdir()])
        ADAPTERS = {}
        inp = Path("/kaggle/input")
        # os.walk with followlinks: Kaggle input folders can be symlinks, and Path.glob("**") skips those
        ALL = sorted(Path(r) / f for r, _, fs in os.walk(inp, followlinks=True) for f in fs) if inp.exists() else []
        print("files under /kaggle/input:", len(ALL))
        for f in ALL[:40]:
            print("  ", f)
        for name in ("main", "control"):
            hits = [f for f in ALL if f.name == "adapter_config.json" and f.parent.name == name]
            if hits:
                ADAPTERS[name] = hits[-1].parent
                continue
            for z in [f for f in ALL if f.suffix == ".zip"]:
                with zipfile.ZipFile(z) as zf:
                    members = [m for m in zf.namelist() if m.startswith(f"adapters/{name}/")]
                    if members and f"adapters/{name}/adapter_config.json" in members:
                        zf.extractall(WORK / "inputs" / z.parent.name, members)
                        ADAPTERS[name] = WORK / "inputs" / z.parent.name / "adapters" / name
                        break
        print("adapters found:", {k: str(v) for k, v in ADAPTERS.items()} or "none (base model only)")
        (REPO / "results/logs/dflash_inputs.json").write_text(json.dumps({"drafter": str(DRAFTER),
            "adapters": {k: str(v) for k, v in ADAPTERS.items()}}, indent=1))
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 8. Both adapters are required here
if STOP is None:
    try:
        missing = [n for n in ("main", "control") if n not in ADAPTERS]
        assert LOCAL_TEST or not missing, f"adapter(s) not attached: {missing}. Add Input -> Your Work -> soup_train_main / soup_train_control"
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. H21 greedy identity (main adapter + plain run in this session), H7 acceptance (control)
if STOP is None:
    try:
        DB = [PY, "scripts/gpu/dflash_bench.py"]
        SM = ["--model", MODEL, "--drafter", DRAFTER, *(["--dtype", "fp32"] if LOCAL_TEST else [])]
        N = ["--n", "4", "--batch", "2", "--max-new", "24"] if LOCAL_TEST else ["--n", "96", "--batch", "32", "--max-new", "512"]
        for tag, plain in (("main", True), ("control", False)):
            if tag not in ADAPTERS:
                print("skip", tag, "(local test without adapters)")
                continue
            run([*DB, "greedy", *SM, "--adapter", ADAPTERS[tag], *N, *(["--plain"] if plain else []),
                 "--ref", f"data/ref_greedy/{tag}.jsonl", "--tag", tag], log=f"dflash_greedy_{tag}.txt", check=False)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 10. H23 batch 48 (plain arm runs with the drafter moved off the GPU)
if STOP is None:
    try:
        B = ["--batches", "4", "--max-new", "16"] if LOCAL_TEST else ["--batches", "48", "--max-new", "512"]
        run([*DB, "sweep", *SM, *B, "--tag", "base48"], log="dflash_sweep48.txt", check=False)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 11. H22 sampling test with the main adapter
if STOP is None:
    try:
        ad = ["--adapter", ADAPTERS["main"]] if "main" in ADAPTERS else []
        S = ["--samples", "60", "--batch", "30", "--dbatch", "30"] if LOCAL_TEST else ["--samples", "2000", "--batch", "48", "--dbatch", "32"]
        run([*DB, "sampling", *SM, *ad, *S, "--tag", "main" if ad else "base_local"], log="dflash_sampling_main.txt", check=False)
    except Exception as e:
        STOP = type(e).__name__ + ': ' + str(e)
        print('STOPPED:', STOP)
else:
    print('skipped because an earlier step failed:', STOP)

In [ ]:
# 9. Peak memory of THIS run (nvidia-smi used minus the ballast) and pack results (always runs)
if not LOCAL_TEST and "smi_log" in globals() and "BALLAST" in globals():
    import csv
    rows = list(csv.reader(open(smi_log.name)))[1:]
    used = [int(r[1].strip().split()[0]) for r in rows if len(r) > 1 and r[1].strip().split()[0].isdigit()]
    # subtract everything that was on the card before our processes started (ballast + driver baseline);
    # pilot 1 subtracted only the ballast and over-reported by the 107 MiB baseline (E12)
    peak = max(used) - BALLAST["smi_used_after_mib"]
    cap = {"smi_peak_total_mib": max(used), "smi_before_us_mib": BALLAST["smi_used_after_mib"],
           "ballast_used_mib": BALLAST["ballast_used_mib"], "our_peak_mib": peak,
           "cap_mib": BALLAST["cap_mib"], "under_cap": peak <= BALLAST["cap_mib"], "samples": len(used)}
    (REPO / "results/logs/cap_summary.json").write_text(json.dumps(cap, indent=2))
    print(cap)
import shutil
(REPO / "results/logs/stop_reason.txt").write_text(str(STOP))
print("STOP reason:", STOP)
zip_base = WORK / "soup_results"
shutil.make_archive(str(zip_base), "zip", REPO / "results")
print("DOWNLOAD:", str(zip_base) + ".zip")